In [10]:


import pandas as pd
import re

from nltk.corpus   import stopwords        # list of useless words (the, is, and...)
from nltk.stem     import WordNetLemmatizer  # converts words to root form
from nltk.tokenize import word_tokenize    # splits sentence into individual words

from sklearn.preprocessing          import LabelEncoder     # converts category names to numbers
from sklearn.feature_extraction.text import TfidfVectorizer  # converts text to numbers
from sklearn.model_selection         import train_test_split  # splits data into train and test
from sklearn.linear_model            import LogisticRegression # the ML model
from sklearn.metrics                 import accuracy_score, classification_report  # checks accuracy

print('All libraries imported!')

All libraries imported!


In [11]:
# Load the CSV file into a dataframe (like an excel table)
df = pd.read_csv('Resume.csv')

# See the shape — how many rows and columns
print('Rows and Columns:', df.shape)

# See the first 3 rows
df.head(3)

Rows and Columns: (2484, 4)


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [12]:
# See how many resumes are in each category
print(df['Category'].value_counts())

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
FINANCE                   118
ADVOCATE                  118
ACCOUNTANT                118
ENGINEERING               118
CHEF                      118
AVIATION                  117
FITNESS                   117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


In [13]:
# Create the lemmatizer and stopwords list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))  # set() makes checking faster

# This function takes one resume text and cleans it
def clean_resume(text):

    # Step 1 — remove website links (http://...)
    text = re.sub(r'http\S+', ' ', text)

    # Step 2 — remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Step 3 — remove @mentions and #hashtags
    text = re.sub(r'[@#]\S+', ' ', text)

    # Step 4 — keep only normal letters, remove everything else
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Step 5 — make everything lowercase
    text = text.lower()

    # Step 6 — split the sentence into individual words
    words = word_tokenize(text)

    # Step 7 — remove stopwords (the, is, and...) and very short words
    words = [w for w in words if w not in stop_words and len(w) > 2]

    # Step 8 — lemmatize each word (running → run, managed → manage)
    words = [lemmatizer.lemmatize(w) for w in words]

    # Step 9 — join words back into one string
    return ' '.join(words)


print('Cleaning all resumes... (takes a minute)')

# Apply the clean function to every resume in the dataset
df['cleaned'] = df['Resume_str'].apply(clean_resume)

print('Done!')

# Compare before and after
print('\nBEFORE cleaning:')
print(df['Resume_str'][0][:200])

print('\nAFTER cleaning:')
print(df['cleaned'][0][:200])

Cleaning all resumes... (takes a minute)
Done!

BEFORE cleaning:
         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Resp

AFTER cleaning:
administrator marketing associate administrator summary dedicated customer service manager year experience hospitality customer service management respected builder leader customer focused team strive


In [14]:
# The model cannot understand text like 'ENGINEERING'
# We need to convert category names → numbers

le = LabelEncoder()

# fit_transform does 2 things:
# 1. learns all unique categories
# 2. converts them to numbers
df['label'] = le.fit_transform(df['Category'])

# See the mapping
print('Category → Number mapping:')
for number, category in enumerate(le.classes_):
    print(f'  {number:2d}  →  {category}')

Category → Number mapping:
   0  →  ACCOUNTANT
   1  →  ADVOCATE
   2  →  AGRICULTURE
   3  →  APPAREL
   4  →  ARTS
   5  →  AUTOMOBILE
   6  →  AVIATION
   7  →  BANKING
   8  →  BPO
   9  →  BUSINESS-DEVELOPMENT
  10  →  CHEF
  11  →  CONSTRUCTION
  12  →  CONSULTANT
  13  →  DESIGNER
  14  →  DIGITAL-MEDIA
  15  →  ENGINEERING
  16  →  FINANCE
  17  →  FITNESS
  18  →  HEALTHCARE
  19  →  HR
  20  →  INFORMATION-TECHNOLOGY
  21  →  PUBLIC-RELATIONS
  22  →  SALES
  23  →  TEACHER


In [15]:
# The model cannot read words — it needs numbers
# TF-IDF converts each resume into a row of numbers
# Each number = how important a word is in that resume

tfidf = TfidfVectorizer(
    max_features=10000,   # use only top 10000 most important words
    ngram_range=(1, 2),   # use single words AND word pairs
    stop_words='english'  # ignore common english words
)

# fit_transform — learns vocabulary AND converts text to numbers
X = tfidf.fit_transform(df['cleaned'])

# y is the target — what we want to predict
y = df['label']

print('Text converted to number matrix!')
print('Matrix shape:', X.shape)
print(f'  {X.shape[0]} resumes')
print(f'  {X.shape[1]} word features')

Text converted to number matrix!
Matrix shape: (2484, 10000)
  2484 resumes
  10000 word features


In [16]:
# Split data into 2 parts:
# Training set (80%) → model learns from this
# Test set     (20%) → we check accuracy on this

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% goes to test
    random_state=42,    # same split every time
    stratify=y          # keep class balance in both sets
)

print('Training samples :', X_train.shape[0])
print('Testing  samples :', X_test.shape[0])

Training samples : 1987
Testing  samples : 497


In [17]:
# Create the Logistic Regression model
model = LogisticRegression(
    max_iter=1000,           # give model enough iterations to learn
    class_weight='balanced'  # treat all categories fairly
)

# Train the model — this is where actual learning happens
print('Training the model...')
model.fit(X_train, y_train)
print('Training done!')

Training the model...
Training done!


In [18]:
# Use the trained model to predict on test data
y_pred = model.predict(X_test)

# Compare predictions vs actual labels
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}  ({accuracy*100:.2f}%)')

# Detailed report — precision, recall, f1 for each category
print('\nDetailed Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.6600  (66.00%)

Detailed Report:
                        precision    recall  f1-score   support

            ACCOUNTANT       0.65      0.83      0.73        24
              ADVOCATE       0.45      0.42      0.43        24
           AGRICULTURE       0.64      0.54      0.58        13
               APPAREL       0.57      0.21      0.31        19
                  ARTS       0.56      0.43      0.49        21
            AUTOMOBILE       0.57      0.57      0.57         7
              AVIATION       0.85      0.71      0.77        24
               BANKING       0.94      0.65      0.77        23
                   BPO       0.50      0.50      0.50         4
  BUSINESS-DEVELOPMENT       0.50      0.83      0.62        24
                  CHEF       0.85      0.71      0.77        24
          CONSTRUCTION       0.85      0.77      0.81        22
            CONSULTANT       0.38      0.13      0.19        23
              DESIGNER       0.85      0.81      0.83     

In [19]:
# Write any resume text here
new_resume = """
Experienced Python developer with 5 years of experience.
Worked with AWS, Docker, and PostgreSQL.
Built REST APIs using Django and FastAPI.
Led a team of 4 engineers on microservices projects.
"""

# Step 1 — clean the new resume using same function
cleaned = clean_resume(new_resume)

# Step 2 — convert to numbers using same tfidf
# NOTE: use transform() NOT fit_transform()
# because we already learned the vocabulary in step 6
numbers = tfidf.transform([cleaned])

# Step 3 — predict
prediction = model.predict(numbers)[0]

# Step 4 — convert number back to category name
category = le.inverse_transform([prediction])[0]

print(f'Predicted Job Category: {category}')

# Step 5 — show confidence for top 3 categories
probabilities = model.predict_proba(numbers)[0]

# Get top 3 categories with highest probability
import numpy as np
top3_index = probabilities.argsort()[-3:][::-1]

print('\nTop 3 Predictions:')
for i in top3_index:
    cat  = le.classes_[i]
    prob = probabilities[i]
    bar  = '█' * int(prob * 30)
    print(f'  {cat:30s}  {prob*100:.1f}%  {bar}')

Predicted Job Category: ENGINEERING

Top 3 Predictions:
  ENGINEERING                     11.2%  ███
  AGRICULTURE                     7.5%  ██
  CONSULTANT                      6.8%  ██
